# Crimes Database

### TOC

### Intro

In this guided project, we will demonstrate the fundamentals of PostgreSQL by building a functional database. The structure will include a defined schema and a primary table, which we will populate by loading data from a `data.csv` file. Additionally, we will illustrate how to define two distinct user groups and assign specific users to each. 

**Rough Project Overview:**

![project overview](project-overview.png)

### Setup

To begin, we will establish a connection using the specific Postgres user created for this project. Once connected, we will execute the commands necessary to initialize our database and schema.

In [1]:
#import psycopg2

#conn = psycopg2.connect(
#    dbname="dq", 
#    user="dq",         # user created for project purposes only
#    password="abc123",
#    host="localhost"
#)
#conn.autocommit = True
#cur = conn.cursor()
#cur.execute("CREATE DATABASE crimes_db;")
#conn.close()

In [2]:
#conn = psycopg2.connect(
#    dbname="crimes_db", 
#    user="dq",         # user created for project purposes only
#    password="abc123",
#    host="localhost"
#)
#conn.autocommit = True
#cur = conn.cursor()
#cur.execute("CREATE SCHEMA crimes;")

### Data

With the database and schema prepared, we are now ready to process our information. We will begin by using the `csv` module to import and read the raw data.

In [3]:
import csv
with open('boston.csv') as file:
    reader = csv.reader(file)
    col_headers = next(reader)
    first_row = next(reader)

print(col_headers)
print(first_row)

['incident_number', 'offense_code', 'description', 'date', 'day_of_the_week', 'lat', 'long']
['1', '619', 'LARCENY ALL OTHERS', '2018-09-02', 'Sunday', '42.35779134', '-71.13937053']


### Data Types

Before migrating the data into our schema’s table, we will analyze the data types of each column to ensure the database is as optimized as possible. To facilitate this, we will define a custom function that compiles a set of all unique values within a specific column. By examining these unique values, we can accurately determine the most appropriate data type for every field.

In [4]:
def get_col_set(csv_file, col_index):
    values = set()
    with open(csv_file, 'r') as f:
        next(f)
        reader = csv.reader(f)
        for row in reader:
            values.add(row[col_index])
    return values

for i in range(len(col_headers)):
    values = get_col_set("boston.csv", i)
    print(col_headers[i], len(values), sep='\t')

incident_number	298329
offense_code	219
description	239
date	1177
day_of_the_week	7
lat	18177
long	18177


Now that we know how many unique values there are in each column, we want to focus on the two columns that contain textual data, the `description` and `day_of_the_week` columns. Our purpose in looking at these two columns, is to identify the longest possible string of text, which will then help us to determine the appropriate datatype we should set to the column. We can see above that the `day_of_the_week`column contains only seven unique values. This tells us that the seven different values are the names of the different days of the week, in which case, we know that "Wednesday" would be the longest. The `description` column, has 239 unique values. To determine how long the longest string is, we will use our `get_col_set()` custom function to make a set of all the unique strings, then compute the maximum length of the strings in the set.

In [5]:
descriptions = get_col_set("boston.csv", 2)
longest_string = max(descriptions, key=len) 
print(longest_string)
print(len(longest_string))

RECOVERED - MV RECOVERED IN BOSTON (STOLEN OUTSIDE BOSTON)
58


With the output from the `description` column above, we can now decide what datatypes we want to assign to each column in our table. As a reminder, our columns are as follows:

- `incident_number`
- `offense_code`
- `description`
- `date`
- `day_of_the_week`
- `lat`
- `long`

For the `incident_number` and `offense_code` columns, we decide to set them to the `INTEGER` type, as they contain numeric data. For the `description` column, we decide to set it to the `VARCHAR(100)` type, as this type will be large enought to comfortably fit the largest sizes of strings while not taking up too much memory space. The `date` column we will set to the `DATE` datatype, and the `lat` and `long` columns we will set to the `decimal` datatype. The `day_of_the_week` column we decide to enumerate due to it's small amount of unique values. We will create our table and datatypes in the code below.